# Подробный обзор проекта: Прогнозирование спроса на такси в Нью-Йорке

Этот раздел содержит детальный разбор архитектуры данных, этапов предобработки, логики построения моделей и их интеграции в бизнес-логику. Пайплайн представляет собой законченное ML-решение: от сырых логов до симуляции работы рекомендательного API.

## Часть 1: Конструирование признаков и профилирование зон (Шаги 1–2)

**Что происходит:** Подготовка сырых данных о поездках к пространственному анализу.  
**Инструменты:** `pandas`, `numpy`.  
**Данные:** Очищенный датасет в формате `.parquet`, содержащий время посадки, координаты, ID зон и погодные условия.

1. **Извлечение времени:** Из базового поля `tpep_pickup_datetime` извлекаются ключевые временные признаки: час (`pickup_hour`), день недели (`day_of_week`) и календарная дата. Они служат основой для выявления циклических паттернов спроса.
2. **Поведенческий профиль (Smart Feature Engineering):** Вместо кластеризации районов Нью-Йорка только по географическим координатам (широте и долготе), собирается их «ритм жизни»:
    * Вычисляются медианные координаты для каждой зоны.
    * Выделяются утренние (7:00–10:00) и вечерние (17:00–20:00) часы пик.
    * Рассчитываются **доли** поездок в эти периоды (`morning_ratio`, `evening_ratio`). 
    
*Переход от абсолютных значений спроса к относительным долям критически важен: это позволяет алгоритму находить функционально похожие зоны (например, два спальных района), даже если один из них генерирует 10 000 поездок, а другой — всего 500.*

## Часть 2: Пространственно-поведенческая кластеризация (Шаги 3–5)

**Что происходит:** Снижение размерности пространства признаков и борьба с шумом в данных (разреженностью спроса в непопулярных локациях).  
**Инструменты:** `sklearn.cluster.KMeans`, `sklearn.preprocessing.StandardScaler`, `sklearn.decomposition.PCA`, `matplotlib`, `seaborn`.

1. **Масштабирование данных:** Перед кластеризацией применяется `StandardScaler`. Поскольку алгоритмы, основанные на расстоянии (K-Means) и дисперсии (PCA), чувствительны к масштабу осей, этот шаг предотвращает доминирование признаков с большими числовыми значениями над долями.
2. **Поиск оптимального числа кластеров ($k$):** Обучение моделей K-Means в цикле от $k=3$ до $k=11$ и оценка качества по двум метрикам:
    * **Inertia (Внутрикластерная дисперсия):** Оценка плотности группировки точек вокруг центроидов (поиск точки излома или «локтя»).
    * **Silhouette Score:** Метрика обособленности кластеров, показывающая, насколько объект ближе к своему кластеру, чем к чужим.
3. **Применение PCA (Метод главных компонент):** Профиль зоны содержит много метрик, и визуализировать его напрямую в 2D-пространстве невозможно. PCA сжимает данные до двух главных компонент, которые сохраняют **65.2%** исходной дисперсии, что позволяет наглядно отобразить структуру сформированных кластеров на графике.

## Часть 3: Формирование таргета и валидация (Шаги 6–7)

**Что происходит:** Переход от логов индивидуальных поездок к непрерывному формату временных рядов (Time Series).

1. **Агрегация (Groupby):** Датасет группируется по составному ключу «Дата + Час + Зона». Количество строк, попавших в каждую группу, формирует целевую переменную — суммарный спрос ($y$).
2. **Объединение матриц:** К целевой переменной по ключам подтягиваются признаки кластеров, к которым относятся зоны, а также внешние погодные факторы (температура, осадки, категориальный код погоды). Таким образом формируется итоговая матрица признаков $X$.
3. **Хронологическое разбиение (Train/Test Split):** Применяется строгая валидация для временных рядов: последние 7 дней данных отрезаются под тест, а всё, что было до них, идет в обучение. Случайное перемешивание (Random Split) здесь недопустимо, так как оно привело бы к утечке данных из будущего (data leakage).

## Часть 4: Обучение моделей и оценка качества (Шаги 8–11)

**Что происходит:** Решение задачи регрессии (предсказание спроса). Сравнение базового алгоритма с продвинутым ансамблем.  
**Модели:** `RandomForestRegressor`, `CatBoostRegressor`.  
**Метрики:** MAE (Средняя абсолютная ошибка), RMSE (Корень из среднеквадратичной ошибки).

1. **Бэйзлайн (Random Forest):** Быстрая модель из 50 деревьев. Показывает базовое качество (MAE ~3.27), но имеет фундаментальное ограничение: она не умеет эффективно работать с высокоразмерными категориальными признаками (такими как ID зоны или код погоды), воспринимая их как непрерывные числа.
2. **Градиентный бустинг (CatBoost):** Применяется последовательное построение ансамбля деревьев, где каждое новое дерево минимизирует ошибку предыдущих.
    * Категориальные признаки явно передаются в параметр `cat_features`, что запускает встроенный механизм Target Encoding.
    * Обучение контролируется параметром `early_stopping_rounds=50` на валидационной выборке во избежание переобучения.
3. **Анализ результатов:** Переход к CatBoost снижает MAE незначительно (на **1.7%**), однако метрика RMSE падает сразу на **13.2%**. Поскольку RMSE штрафует за крупные ошибки квадратично, это доказывает, что бустинг гораздо точнее предсказывает пиковые нагрузки в часы пик и практически ликвидировал аномально сильные промахи в прогнозах.

## Часть 5: Бизнес-применение и интерпретация (Шаги 12–13)

**Что происходит:** Извлечение инсайтов из модели и написание симулятора продуктовой логики.

1. **Feature Importance (Важность признаков):** Анализ внутренней структуры модели показывает, что ключевыми факторами для прогноза являются час поездки (`pickup_hour`) и географический идентификатор зоны (`PULocationID`). Погодные условия выступают в роли корректирующих фильтров второго порядка.
2. **Разработка сервисной функции `get_best_zones_catboost`:** Написана функция, имитирующая бэкенд-логику мобильного приложения для водителей:
    * Принимает текущий контекст: час, день недели и погодные условия.
    * Генерирует тестовый вектор для всех доступных зон Нью-Йорка.
    * Запускает инференс модели CatBoost.
    * Корректирует невозможные отрицательные прогнозы с помощью `np.maximum(0, expected_demand)`.
    * Сортирует зоны по убыванию спроса и возвращает топ-5 лучших локаций для отправки туда свободных экипажей такси.